# Randomness dan Reproducibility dalam Machine Learning

Target Belajar:

* Mendefinisikan konsep randomness dalam konteks machine learning secara presisi
* Mengidentifikasi titik-titik dalam pipeline ML di mana keacakan muncul
* Menjelaskan mekanisme kerja random_state sebagai seed generator bilangan pseudo-acak
* Membedakan perilaku random_state saat diisi int, RandomState instance, dan None
*  Menjelaskan signifikansi ilmiah dari reproducibility, bukan sekadar kebiasaan teknis
* Mengidentifikasi kapan integer seed cukup, dan kapan justru berisiko menyembunyikan ketidakstabilan model (poin lanjutan, berbasis dokumentasi resmi)

# Bagian 1: Penjelasan Konsep

## 1.1 Definisi Randomness dalam Machine Learning

* Randomness (keacakan): dalam ML merujuk pada komponen algoritmik yang hasilnya bergantung pada bilangan acak, bukan murni fungsi deterministik dari input
* Randomness bukan cacat desain — pada banyak algoritma, randomness adalah bagian esensial dari mekanisme kerja, bukan efek samping yang tidak diinginkan.
* Randomness dalam komputer bersifat pseudo-random, bukan acak murni:
    * Dihasilkan oleh algoritma deterministik yang disebut Pseudo-Random Number Generator (PRNG)
    * Scikit-learn dan NumPy menggunakan varian algoritma Mersenne Twister (MT19937) sebagai PRNG default
    * PRNG membutuhkan titik awal, disebut seed — dari seed yang sama, PRNG akan selalu menghasilkan urutan bilangan "acak" yang identik
* Implikasi mendasar: keacakan pada ML sebenarnya bisa dikontrol sepenuhnya, karena akar dari keacakan itu bukan proses fisik acak, melainkan seed numerik.

## 1.2 Titik-Titik Munculnya Randomness dalam Pipeline ML

| Tahap Pipeline                        | Sumber Randomness                                                       | Materi Terkait                   |
| ------------------------------------- | ----------------------------------------------------------------------- | -------------------------------- |
| Split data                            | Pengacakan urutan baris sebelum dibagi train/test                       | `02_train_test_split`            |
| Inisialisasi model linear (SGD-based) | Bobot awal (*weight initialization*) sebelum optimisasi                 | Fase 5 (Deep Learning)           |
| Decision Tree / Random Forest         | Pemilihan subset fitur & subset sampel (*bootstrap*) di tiap split node | Fase 3.3                         |
| K-Means                               | Posisi centroid awal                                                    | Fase 3.4 (Unsupervised Learning) |
| Cross-validation                      | Pembagian data ke tiap fold                                             | Fase 3.6                         |
| SMOTE / oversampling                  | Pemilihan tetangga untuk interpolasi sampel sintetis                    | Fase 3.7                         |
| Neural Network                        | Inisialisasi bobot layer, dropout mask, shuffling batch                 | Fase 5                           |


> Poin kunci: setiap kali suatu algoritma memiliki parameter random_state di scikit-learn, itu adalah sinyal eksplisit bahwa algoritma tersebut memiliki komponen internal yang bergantung pada bilangan acak.

## 1.3 Mekanisme Kerja random_state

* random_state: parameter yang mengontrol seed dari PRNG yang dipakai estimator/fungsi tersebut
* Berdasarkan dokumentasi resmi, random_state dapat diisi tiga jenis nilai, dengan perilaku berbeda:

(a) random_state=None (default)

* Estimator memakai instance RandomState global dari np.random
* Setiap pemanggilan fungsi akan menghasilkan hasil yang berbeda, karena state generator global terus berubah/maju setiap dipakai
* Konsekuensi: hasil eksperimen tidak dapat direplikasi dari sesi ke sesi

(b) random_state=<integer> (misal 42)

* Integer berfungsi sebagai seed - titik awal deterministik untuk PRNG
* Setia pemanggilan dengan seed yang sama akan  menghasilkan urutan bilangan acak yang identik
* Ini adalah opsi paling umum dipakai untuk kebutuhan reproducibility dasar

(c) random_state=<np.random.RandomState instance>

* Sebuah objek generator yang sudah diinisialisasi diberikan langsung ke estimator
* Estimator akan memakai dan memajukan state generator tersebut, bukan membuat generator baru dari seed
* Relevansi lanjutan: dibahas di bagian 1.5 

## 1.4 Reproducibility

Reproducibility (kemampuan mereplikasi hasil eksperimen secara identik) memiliki fondasi metodologis, bukan sekadar preferensi gaya penulisan kode:

* Validitas ilmiah: dalam metode ilmiah, sebuah klaim (termasuk klaim "model saya mencapai akurasi 92%") harus dapat diverifikasi ulang oleh pihak lain dengan prosedur yang sama. Tanpa random_state tetap, verifikasi ini tidak mungkin dilakukan secara presisi.
* Debugging sistematis: ketika kode ML menghasilkan output yang tidak sesuai ekspektasi, proses debugging mengharuskan kondisi yang identik di setiap percobaan ulang. Tanpa seed tetap, developer tidak bisa membedakan apakah perubahan hasil berasal dari perubahan kode atau variasi acak semata.
* Perbandingan model yang adil (fair comparison): ketika membandingkan performa Model A vs Model B (topik materi 15_model_comparison_experiment), kedua model harus dievaluasi pada split data dan kondisi acak yang identik — jika tidak, perbedaan skor bisa jadi murni disebabkan oleh perbedaan split, bukan oleh kualitas model itu sendiri.
* Kolaborasi & audit: pada lingkungan kerja tim maupun proses audit/compliance (relevan pada materi Fairness di 3.11), pihak lain harus bisa menjalankan ulang pipeline dan mendapat hasil yang sama persis untuk keperluan verifikasi.

## 1.5 Nuansa Lanjutan: Integer vs RandomState pada Cross-Validation

Saat mengevaluasi estimator ber-randomness (misal SGDClassifier, RandomForestClassifier) menggunakan cross-validation, terdapat dua kemungkinan pendekatan terhadap random_state estimator tersebut:

### Pendekatan 1 — Integer tetap

(misal random_state=42) diberikan ke estimator:

* Estimator akan menggunakan RNG (random number generator) yang identik di setiap fold cross-validation
* Risiko: jika performa model kebetulan baik (atau buruk) akibat inisialisasi acak tertentu, kondisi itu akan terulang secara identik di semua fold — sehingga skor CV tidak mendeteksi ketidakstabilan model terhadap inisialisasi acaknya sendiri

### Pendekatan 2 — RandomState instance

None diberikan ke estimator:

* Estimator akan menggunakan RNG yang berbeda di tiap fold
* Efeknya: variansi skor antar fold mencerminkan dua sumber ketidakpastian sekaligus — variasi karena data yang berbeda di tiap fold, dan variasi karena inisialisasi acak model itu sendiri
* Berdasarkan dokumentasi resmi mengenai common pitfalls, disebutkan bahwa untuk mengevaluasi performa estimator dengan cross-validation, evaluasi ingin memastikan estimator dapat memberi prediksi akurat untuk data baru, sekaligus memastikan estimator robust terhadap inisialisasi acaknya — misalnya inisialisasi bobot acak pada SGDClassifier diharapkan konsisten baik di semua fold, karena jika tidak, saat dilatih pada data baru estimator bisa jadi kurang beruntung dan menghasilkan performa buruk.    
* Rekomendasi resmi terkait praktik ini: untuk robustness optimal hasil cross-validation, sebaiknya memberikan RandomState instance saat membuat estimator, atau membiarkan random_state bernilai None — sedangkan untuk CV splitter (seperti KFold), memberikan integer justru merupakan opsi paling aman dan lebih disarankan.

### Kesimpulan praktis

* Untuk CV splitter (KFold, StratifiedKFold, train_test_split) → integer seed disarankan (demi reproducibility split)
* Untuk estimator ber-randomness internal yang dievaluasi lewat CV → mempertimbangkan RandomState instance atau None bisa memberi gambaran yang lebih jujur soal stabilitas model, meski trade-off-nya adalah hasil individual run menjadi kurang reproducible secara literal
* Ini adalah nuansa lanjutan — untuk kebutuhan belajar dasar (materi 3.1 ini), integer seed tetap merupakan praktik standar yang tepat dan cukup. Poin ini penting untuk disimpan sebagai awareness, dan akan relevan kembali secara detail di materi 3.6 Cross-Validation Lanjutan (Repeated K-Fold).

## 1.6 Ringkasan Prinsip Operasional

* Randomness dalam ML bersifat pseudo, sehingga sepenuhnya dapat dikontrol melalui seed.
* random_state=None → hasil tidak reproducible, tidak disarankan untuk eksperimen yang perlu direplikasi/dibandingkan.
random_state=<int> → praktik standar untuk reproducibility dasar dan perbandingan model yang adil.
* Nilai spesifik integer (0, 1, 42, dst.) tidak memiliki makna matematis khusus — yang penting adalah konsistensi pemakaian nilai yang sama di seluruh eksperimen yang dibandingkan.
* Untuk kasus lanjutan (CV terhadap estimator ber-randomness), pertimbangan integer vs RandomState instance memengaruhi apa yang sebenarnya diukur oleh variansi skor CV.

# Bagian 2: Implementasi (Pembuktian Empiris)

## 2.1 Randomness Bersifat Pseudo — Pembuktian dengan Seed Sama

In [2]:
import numpy as np

# Dua generator terpisah, seed SAMA

rng_a = np.random.RandomState(42)
rng_b = np.random.RandomState(42)

print("Generator A:", rng_a.rand(5))
print("Generator B:", rng_b.rand(5))
print("Identik?", np.array_equal(rng_a.rand(5), rng_b.rand(5)))

Generator A: [0.37454012 0.95071431 0.73199394 0.59865848 0.15601864]
Generator B: [0.37454012 0.95071431 0.73199394 0.59865848 0.15601864]
Identik? True


Perhatikan: 
* dua generator independen, dengan seed yang sama, menghasilkan urutan bilangan yang identik. 
* Ini membuktikan langsung klaim di bagian 1.1 — keacakan komputer sepenuhnya deterministik jika seed diketahui.

## 2.2 Efek random_state=None vs Integer pada train_test_split

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris 
import pandas as pd

In [5]:
data = load_iris()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)


Tanpa random_state - jalankan 2x, index harus berbeda kemungkinan besar

In [17]:
split_1 = train_test_split(X, y, test_size=0.2)
split_2 = train_test_split(X, y, test_size=0.2)


print("None — identik antar run?",
      split_1[0].index.tolist() == split_2[0].index.tolist())

None — identik antar run? False


Dengan random_state=42 — jalankan 2x, index HARUS identik

In [20]:
split_3 = train_test_split(X, y, test_size=0.2, random_state=42)
split_4 = train_test_split(X, y, test_size=0.2, random_state=42)
print("Integer seed — identik antar run?",
      split_3[0].index.tolist() == split_4[0].index.tolist())

Integer seed — identik antar run? True


## 2.3 Dampak Randomness terhadap Skor Model (Tanpa Kontrol Seed)

In [21]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

skor_list = []
for percobaan in range(5):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2   # TANPA random_state — sengaja
    )
    model = DecisionTreeClassifier()   # TANPA random_state — sengaja
    model.fit(X_train, y_train)
    skor = model.score(X_test, y_test)
    skor_list.append(skor)
    print(f"Percobaan {percobaan+1}: akurasi = {skor:.4f}")

print("\nVariasi skor (std):", np.std(skor_list))

Percobaan 1: akurasi = 0.8667
Percobaan 2: akurasi = 0.9667
Percobaan 3: akurasi = 1.0000
Percobaan 4: akurasi = 0.9667
Percobaan 5: akurasi = 0.9667

Variasi skor (std): 0.04521553322083511


* Jalankan sendiri dan perhatikan bahwa skor akurasi dapat berubah-ubah antar percobaan.
* Padahal, kode dan dataset yang digunakan tetap identik.
* Ini merupakan bukti empiris dari risiko reproducibility yang dibahas pada bagian 1.4.
* Tanpa seed tetap, sulit membedakan:

  * model benar-benar membaik, atau
  * hasilnya hanya dipengaruhi split/inisialisasi acak yang kebetulan menguntungkan.


## 2.4 Reproduksi Penuh dengan Seed Tetap

In [22]:
skor_list_fixed = []
for percobaan in range(5):
    X_trian, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)
    skor = model.score(X_test, y_test)
    skor_list_fixed.append(skor)
    print(f"Percobaan {percobaan+1}: Akurasi = {skor:.4f}")

print("\nVariasi skor (std):", np.std(skor_list_fixed))

Percobaan 1: Akurasi = 0.1667
Percobaan 2: Akurasi = 0.1667
Percobaan 3: Akurasi = 0.1667
Percobaan 4: Akurasi = 0.1667
Percobaan 5: Akurasi = 0.1667

Variasi skor (std): 0.0


Dengan seed tetap di kedua titik randomness (split data dan inisialisasi model), skor akan identik persis di setiap percobaan — std akan bernilai 0.0.

## 2.5 Pembuktian Nuansa Lanjutan: Integer vs RandomState Instance pada CV

In [26]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestClassifier

cv = KFold(n_splits=5, shuffle=True, random_state=42)   # splitter: integer OK

# Skenario 1: estimator pakai integer tetap → RNG SAMA di semua fold
model_int = RandomForestClassifier(random_state=42, n_estimators=50)
skor_int = cross_val_score(model_int, X, y, cv=cv)
print("Estimator random_state=int      →", skor_int, "| std:", skor_int.std())

# Skenario 2: estimator pakai None → RNG BERBEDA di tiap fold
model_none = RandomForestClassifier(random_state=None, n_estimators=50)
skor_none = cross_val_score(model_none, X, y, cv=cv)
print("Estimator random_state=None     →", skor_none, "| std:", skor_none.std())

Estimator random_state=int      → [1.         0.96666667 0.93333333 0.93333333 0.96666667] | std: 0.024944382578492935
Estimator random_state=None     → [1.         0.96666667 0.93333333 0.93333333 0.96666667] | std: 0.024944382578492935


* Bandingkan nilai `std` (standar deviasi) pada kedua skenario.
* `random_state=None` pada estimator dapat menghasilkan variasi skor antar percobaan.
* Variasi tersebut dapat mencerminkan sensitivitas model terhadap inisialisasi acaknya sendiri.
* Hal ini sesuai dengan penjelasan pada bagian 1.5.
* Jalankan eksperimen beberapa kali untuk mengamati polanya.


## Kesalahan Umum

| Kesalahan                                                                  | Penjelasan / Dampak                                                                                                                                  |
| -------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------- |
| Tidak mengisi `random_state` sama sekali                                   | Eksperimen tidak dapat direplikasi dan debugging menjadi tidak sistematis                                                                            |
| Mengisi `random_state` berbeda pada tiap model saat membandingkan performa | Perbandingan menjadi tidak adil — perbedaan skor bisa berasal dari split/inisialisasi, bukan kualitas model                                          |
| Menganggap `random_state` hanya relevan untuk `train_test_split`           | Randomness juga muncul pada banyak estimator lain seperti Tree, Forest, K-Means, dan SGD                                                             |
| Menganggap integer seed selalu pilihan "paling benar"                      | Dalam CV dengan estimator yang memiliki randomness, `RandomState` instance atau `None` dapat memberikan estimasi stabilitas yang lebih representatif |
| Mengira nilai `42` memiliki makna matematis khusus                         | Nilai seed bersifat arbitrer — yang penting adalah konsistensi penggunaannya                                                                         |


# Bagian 3: Latihan

```text
Latihan 1 — Identifikasi Titik Randomness
Dari tabel di bagian 1.2, pilih 2 algoritma (boleh dari materi yang belum dipelajari, cek dokumentasi). Cari dan catat: apakah algoritma tersebut punya parameter random_state? Apa yang menjadi objek keacakannya (dijelaskan di docstring)?

Latihan 2 — Reproduksi Eksperimen Bagian 2.3
Jalankan ulang kode di bagian 2.3 sebanyak 10 percobaan (bukan 5), lalu hitung std skornya. Bandingkan dengan hasil di bagian 2.4 (seed tetap). Tulis satu simpulan mengenai hubungan antara jumlah sumber randomness yang tidak dikontrol dengan besarnya variansi skor.

Latihan 3 — Uji Konsistensi antar Sesi
Tutup dan buka ulang notebook/kernel Python kamu. Jalankan ulang kode bagian 2.2 (dengan random_state=42). Apakah hasil index split tetap identik dengan sesi sebelumnya? Jelaskan kaitannya dengan konsep PRNG deterministik di bagian 1.1.

Latihan 4 — Eksperimen Nuansa Lanjutan
Jalankan kode di bagian 2.5 sebanyak 3 kali berturut-turut untuk masing-masing skenario. Amati: apakah skor model_int (random_state=42 tetap) selalu identik di setiap run program? Apakah skor model_none selalu identik? Jelaskan hasilnya berdasarkan konsep di bagian 1.3 dan 1.5.

Latihan 5 — Refleksi Tertulis (Sistematis)
Jawab dalam bentuk poin, bukan paragraf:

Definisikan PRNG dan seed dengan kalimat sendiri
Sebutkan minimal 3 titik randomness dalam pipeline ML selain train_test_split
Jelaskan risiko metodologis dari membandingkan dua model dengan random_state yang berbeda
Jelaskan kapan integer seed pada estimator bisa menyembunyikan ketidakstabilan model, berdasarkan bagian 1.5